# 01 — Occurrence data preparation

Reproducible preparation of *Sapajus libidinosus* occurrence records downloaded from GBIF.

This notebook:
1. locates and reads the raw GBIF download;
2. validates the expected fields;
3. audits taxonomy, occurrence status, country and coordinates;
4. creates a spatial GeoDataFrame;
5. removes exact coordinate duplicates deterministically, prioritizing records with lower reported coordinate uncertainty;
6. exports the cleaned occurrence dataset and an audit log.

Environmental filtering is intentionally left for `02_environment.ipynb`.


In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Project directories
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_DIR.resolve())
print("Processed data:", PROCESSED_DIR.resolve())


## 1. Locate the raw GBIF download


In [ ]:
# WorldClim files are stored elsewhere in the project, but this exclusion
# prevents an environmental ZIP from being mistaken for the GBIF download.
gbif_zips = [
    path for path in RAW_DIR.glob("*.zip")
    if "wc2.1" not in path.name.lower()
]

print(f"Candidate GBIF ZIPs: {len(gbif_zips)}")
for path in gbif_zips:
    print(" -", path.name)

if len(gbif_zips) != 1:
    raise RuntimeError(
        f"Expected exactly 1 GBIF ZIP in {RAW_DIR}, "
        f"but found {len(gbif_zips)}."
    )

gbif_zip = gbif_zips[0]
print("\nSelected:", gbif_zip.name)


## 2. Read the occurrence table directly from the ZIP


In [ ]:
with zipfile.ZipFile(gbif_zip, "r") as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    print("CSV files inside ZIP:", csv_files)

    if len(csv_files) != 1:
        raise RuntimeError(
            f"Expected exactly 1 CSV in the GBIF ZIP, "
            f"but found {len(csv_files)}."
        )

    with z.open(csv_files[0]) as f:
        raw = pd.read_csv(
            f,
            sep="\t",
            low_memory=False
        )

print(f"Records: {len(raw)}")
print(f"Columns: {len(raw.columns)}")


## 3. Select and validate fields used by the occurrence pipeline


In [ ]:
COLUMNS = [
    "gbifID",
    "species",
    "scientificName",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "year",
    "basisOfRecord",
    "occurrenceStatus",
    "countryCode",
    "stateProvince",
    "locality",
    "datasetKey",
    "institutionCode",
    "issue"
]

missing_columns = [col for col in COLUMNS if col not in raw.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

occ = raw[COLUMNS].copy()

print("Selected columns:", len(occ.columns))
display(occ.head())


## 4. Initial audit


In [ ]:
audit_summary = pd.Series({
    "raw_records": len(occ),
    "unique_species": occ["species"].nunique(dropna=False),
    "present_records": occ["occurrenceStatus"].eq("PRESENT").sum(),
    "brazil_records": occ["countryCode"].eq("BR").sum(),
    "missing_latitude": occ["decimalLatitude"].isna().sum(),
    "missing_longitude": occ["decimalLongitude"].isna().sum(),
    "missing_uncertainty": occ["coordinateUncertaintyInMeters"].isna().sum(),
    "missing_year": occ["year"].isna().sum()
}, name="value")

display(audit_summary.to_frame())


In [ ]:
# These assertions make the notebook stop if the raw download no longer
# matches the dataset we intend to model.
assert occ["species"].eq("Sapajus libidinosus").all(), (
    "Unexpected taxon found in the GBIF download."
)
assert occ["occurrenceStatus"].eq("PRESENT").all(), (
    "At least one record is not marked PRESENT."
)
assert occ["countryCode"].eq("BR").all(), (
    "At least one record is not assigned to Brazil."
)

print("Taxon, occurrence status and country checks: OK")


### Record types and data sources


In [ ]:
print("Basis of record:")
display(
    occ["basisOfRecord"]
    .value_counts(dropna=False)
    .rename("n")
    .to_frame()
)

print("\nInstitution:")
display(
    occ["institutionCode"]
    .value_counts(dropna=False)
    .rename("n")
    .to_frame()
)

print("\nNumber of datasets:", occ["datasetKey"].nunique(dropna=True))


### Coordinate uncertainty


In [ ]:
print("Coordinate uncertainty (m):")
display(occ["coordinateUncertaintyInMeters"].describe())

print(
    "\nMissing uncertainty:",
    occ["coordinateUncertaintyInMeters"].isna().sum()
)


### GBIF processing issues


In [ ]:
issues_count = (
    occ["issue"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .value_counts()
    .rename_axis("issue")
    .reset_index(name="n")
)

display(issues_count)


The GBIF `issue` field is retained for later methodological decisions.  
This notebook does **not** discard a record merely because it has a GBIF processing flag.


## 5. Validate geographic coordinates


In [ ]:
valid_coordinates = (
    occ["decimalLatitude"].notna()
    & occ["decimalLongitude"].notna()
    & occ["decimalLatitude"].between(-90, 90)
    & occ["decimalLongitude"].between(-180, 180)
)

n_before_coordinates = len(occ)
n_invalid_coordinates = (~valid_coordinates).sum()

print("Records before coordinate validation:", n_before_coordinates)
print("Invalid/missing coordinates:", n_invalid_coordinates)

occ_valid_coords = occ.loc[valid_coordinates].copy()

print("Records after coordinate validation:", len(occ_valid_coords))


## 6. Create the spatial dataset


In [ ]:
gdf = gpd.GeoDataFrame(
    occ_valid_coords,
    geometry=gpd.points_from_xy(
        occ_valid_coords["decimalLongitude"],
        occ_valid_coords["decimalLatitude"]
    ),
    crs="EPSG:4326"
)

print("CRS:", gdf.crs)
print("Spatial records:", len(gdf))
display(gdf.head())


## 7. Diagnose exact coordinate duplication


In [ ]:
n_total = len(gdf)

n_unique_coordinates = (
    gdf[["decimalLatitude", "decimalLongitude"]]
    .drop_duplicates()
    .shape[0]
)

n_duplicates = n_total - n_unique_coordinates

print("Records:", n_total)
print("Unique exact coordinates:", n_unique_coordinates)
print("Exact-coordinate duplicates:", n_duplicates)
print(f"Duplicate percentage: {n_duplicates / n_total * 100:.1f}%")


In [ ]:
duplicate_locations = (
    gdf.groupby(
        ["decimalLatitude", "decimalLongitude"],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .query("n > 1")
    .sort_values("n", ascending=False)
)

display(duplicate_locations.head(20))


## 8. Deterministic exact-coordinate deduplication

For records sharing the same latitude and longitude, the retained record is chosen by:

1. smallest reported `coordinateUncertaintyInMeters`;
2. records with missing uncertainty are ranked after records with known uncertainty;
3. `gbifID` is used as a deterministic tie-breaker.

This removes repeated evidence at the exact same coordinate while preserving the best available spatial metadata.


In [ ]:
gdf_unique = gdf.copy()

gdf_unique["_uncertainty_sort"] = (
    gdf_unique["coordinateUncertaintyInMeters"]
    .fillna(np.inf)
)

# String conversion makes the tie-break robust to identifier dtype.
gdf_unique["_gbif_sort"] = gdf_unique["gbifID"].astype(str)

gdf_unique = (
    gdf_unique
    .sort_values(
        ["_uncertainty_sort", "_gbif_sort"],
        ascending=[True, True],
        kind="stable"
    )
    .drop_duplicates(
        subset=["decimalLatitude", "decimalLongitude"],
        keep="first"
    )
    .drop(columns=["_uncertainty_sort", "_gbif_sort"])
    .reset_index(drop=True)
)

print("Records after exact-coordinate deduplication:", len(gdf_unique))


## 9. Post-deduplication audit


In [ ]:
print("=== CLEAN OCCURRENCE DATASET ===")
print("Records:", len(gdf_unique))
print(
    "Missing coordinate uncertainty:",
    gdf_unique["coordinateUncertaintyInMeters"].isna().sum()
)

print("\nCoordinate uncertainty (m):")
display(gdf_unique["coordinateUncertaintyInMeters"].describe())


In [ ]:
# Reproducibility checks based on the current GBIF download used in this project.
EXPECTED_RAW = 728
EXPECTED_UNIQUE = 606

assert len(raw) == EXPECTED_RAW, (
    f"Raw record count changed: expected {EXPECTED_RAW}, got {len(raw)}."
)
assert len(gdf_unique) == EXPECTED_UNIQUE, (
    f"Unique-coordinate count changed: expected {EXPECTED_UNIQUE}, "
    f"got {len(gdf_unique)}."
)

print("Expected-count checks: OK")


## 10. Audit log


In [ ]:
audit_01 = pd.DataFrame([
    {
        "step": "GBIF raw download",
        "n_records": len(raw),
        "removed_at_step": 0
    },
    {
        "step": "Valid geographic coordinates",
        "n_records": len(occ_valid_coords),
        "removed_at_step": len(raw) - len(occ_valid_coords)
    },
    {
        "step": "Unique exact coordinates",
        "n_records": len(gdf_unique),
        "removed_at_step": len(occ_valid_coords) - len(gdf_unique)
    }
])

display(audit_01)


## 11. Export reproducible outputs


In [ ]:
occurrence_output = PROCESSED_DIR / "sapajus_occurrences_unique.gpkg"
audit_output = PROCESSED_DIR / "audit_01_occurrences.csv"

gdf_unique.to_file(
    occurrence_output,
    driver="GPKG"
)

audit_01.to_csv(
    audit_output,
    index=False
)

print("Saved occurrence dataset:")
print(occurrence_output.resolve())

print("\nSaved audit log:")
print(audit_output.resolve())


## Output of this notebook

The next stage should use:

- `data/processed/sapajus_occurrences_unique.gpkg`
- `data/processed/audit_01_occurrences.csv`

Environmental extraction, environmental NoData filtering, raster-cell redundancy, calibration area/background selection and spatial validation belong to subsequent stages and are intentionally not performed here.
